In [ ]:
# ============================================================
# Helmet + Two-Wheeler ROI Inference  (Pillion / Double-Riding Fix)
# ============================================================
# Changes vs previous version:
#  1. pillion_margin_above (default 120 px) — helmet center is
#     allowed to be this many pixels ABOVE the vehicle box top
#     edge, as long as its X is within the vehicle X range.
#     Fixes the pillion rider whose head sits higher than the
#     expanded vehicle box.
#  2. max_helmets_per_vehicle (default 2) — up to 2 helmets
#     per vehicle track are kept (front + pillion rider).
#     Previously only the single highest-scoring DINO box
#     survived; the second helmet was silently dropped.
#  3. match against ALL live vehicle_tracks (not only
#     current_vehicle_ids) so a pillion helmet is still linked
#     even when YOLO temporarily drops the bike for a frame
#     (within the vehicle_max_misses budget).
# ============================================================

!pip install -q ultralytics transformers accelerate pillow opencv-python-headless

from google.colab import drive
drive.mount('/content/drive')

import json
import logging
import os
import shutil
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
from ultralytics import YOLO

# ── paths ────────────────────────────────────────────────────
DATASET_DIR = Path('/content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset')
INPUT_VIDEO = DATASET_DIR / 'laketown_TG_gholaghata_service_road_1min.avi'
OUTPUT_DIR  = DATASET_DIR / 'helmet_vehicle_roi_outputs'

GROUNDINGDINO_MODEL_ID = 'IDEA-Research/grounding-dino-tiny'
YOLO_MODEL_PATH        = 'yolov8s.pt'
PROMPT                 = 'helmet.'

ROI_POINTS = np.array(
    [
        [0,    1215],
        [297,  1004],
        [2064, 1071],
        [2067, 1510],
        [13,   1515],
    ],
    dtype=np.int32,
)

# ── config ───────────────────────────────────────────────────
@dataclass
class Config:
    input_video:            Path  = INPUT_VIDEO
    output_dir:             Path  = OUTPUT_DIR
    groundingdino_model_id: str   = GROUNDINGDINO_MODEL_ID
    yolo_model:             str   = YOLO_MODEL_PATH
    prompt:                 str   = PROMPT
    box_threshold:          float = 0.50
    text_threshold:         float = 0.30
    inference_width:        int   = 896
    min_box_size:           int   = 20
    max_box_size:           int   = 180
    min_aspect:             float = 0.35
    max_aspect:             float = 2.5
    helmet_confirm_frames:  int   = 2
    vehicle_max_misses:     int   = 20
    vehicle_match_distance: float = 220.0
    vehicle_upward_expand:  float = 0.9
    # --- pillion / double-riding fixes ---
    pillion_margin_above:   int   = 120   # px above vehicle box top allowed for pillion helmet
    max_helmets_per_vehicle:int   = 2     # allow front + pillion helmet per track
    # -------------------------------------
    max_seconds:            float = None
    device:                 str   = field(default_factory=lambda: 'cuda' if torch.cuda.is_available() else 'cpu')
    local_files_only:       bool  = False

config = Config()
print(config)

# ── helpers ──────────────────────────────────────────────────
def setup_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger('helmet_vehicle_roi')
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    formatter = logging.Formatter('[%(asctime)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    file_handler = logging.FileHandler(log_path, mode='w', encoding='utf-8')
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    return logger


def to_device(inputs, device: str):
    return {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs.items()}


def post_process_groundingdino(processor, outputs, input_ids, target_sizes, box_threshold, text_threshold):
    try:
        return processor.post_process_grounded_object_detection(
            outputs, input_ids,
            box_threshold=box_threshold, text_threshold=text_threshold,
            target_sizes=target_sizes,
        )[0]
    except TypeError:
        return processor.post_process_grounded_object_detection(
            outputs, input_ids=input_ids,
            threshold=box_threshold, text_threshold=text_threshold,
            target_sizes=target_sizes,
        )[0]


def iou(a, b) -> float:
    xi1, yi1 = max(a[0], b[0]), max(a[1], b[1])
    xi2, yi2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    if inter <= 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    return inter / max(area_a + area_b - inter, 1)


def center_distance(a, b) -> float:
    acx, acy = (a[0] + a[2]) / 2, (a[1] + a[3]) / 2
    bcx, bcy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
    return float(np.hypot(acx - bcx, acy - bcy))


def clamp_box(box, width, height):
    x1, y1, x2, y2 = box
    return [max(0, x1), max(0, y1), min(width, x2), min(height, y2)]


def box_center(box):
    return ((box[0] + box[2]) // 2, (box[1] + box[3]) // 2)


def save_crop(frame, box, directory: Path, label: str, frame_idx: int, score: float, ts: str):
    x1, y1, x2, y2 = box
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    ts_file = ts.replace(':', '-').replace(' ', '_')
    path = directory / f'frame_{frame_idx:06d}_{label}_conf{score:.2f}_{ts_file}.jpg'
    cv2.imwrite(str(path), crop)
    return path


def match_vehicle_track(vehicle_box, tracks, max_center_distance):
    best_id, best_score = None, -1.0
    for track_id, track in tracks.items():
        overlap   = iou(vehicle_box, track['box'])
        distance  = center_distance(vehicle_box, track['box'])
        if overlap > 0.15:
            score = 2.0 + overlap
        elif distance <= max_center_distance:
            score = 1.0 - (distance / max_center_distance)
        else:
            continue
        if score > best_score:
            best_id, best_score = track_id, score
    return best_id


# ── inference ────────────────────────────────────────────────
def run_inference(cfg: Config):
    input_video  = Path(cfg.input_video)
    output_dir   = Path(cfg.output_dir)

    helmet_crop_dir    = output_dir / 'helmet_crops'
    no_helmet_crop_dir = output_dir / 'no_helmet_crops'
    frame_log_dir      = output_dir / 'frame_debug_logs'
    output_video_path  = output_dir / ('output_' + input_video.name)
    debug_log_path     = output_dir / 'debug_log.txt'
    detections_json_path = output_dir / 'detections.json'

    if output_dir.exists():
        shutil.rmtree(output_dir)
    helmet_crop_dir.mkdir(parents=True, exist_ok=True)
    no_helmet_crop_dir.mkdir(parents=True, exist_ok=True)
    frame_log_dir.mkdir(parents=True, exist_ok=True)

    logger = setup_logger(debug_log_path)
    logger.info('========== HELMET + TWO-WHEELER ROI INFERENCE (PILLION FIX) ==========')
    logger.info(f'INPUT VIDEO : {input_video}')
    logger.info(f'OUTPUT DIR  : {output_dir}')
    logger.info(f'DEVICE      : {cfg.device}')
    logger.info(f'DINO MODEL  : {cfg.groundingdino_model_id}')
    logger.info(f'YOLO MODEL  : {cfg.yolo_model}')
    logger.info(f'ROI POINTS  : {ROI_POINTS.tolist()}')
    logger.info(f'pillion_margin_above    : {cfg.pillion_margin_above}')
    logger.info(f'max_helmets_per_vehicle : {cfg.max_helmets_per_vehicle}')
    logger.info(f'MAX SECONDS : {cfg.max_seconds}')

    logger.info('Loading GroundingDINO...')
    processor = AutoProcessor.from_pretrained(cfg.groundingdino_model_id, local_files_only=cfg.local_files_only)
    model     = AutoModelForZeroShotObjectDetection.from_pretrained(cfg.groundingdino_model_id, local_files_only=cfg.local_files_only)
    model.to(cfg.device).eval()

    logger.info('Loading YOLO...')
    yolo_model = YOLO(str(cfg.yolo_model))
    logger.info('Models loaded.')

    cap = cv2.VideoCapture(str(input_video))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open video: {input_video}')

    fps          = cap.get(cv2.CAP_PROP_FPS) or 25
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_frames   = min(total_frames, int(round(fps * cfg.max_seconds))) if cfg.max_seconds else total_frames

    logger.info(f'VIDEO: {width}x{height} @ {fps:.2f}fps, total_frames={total_frames}')
    logger.info(f'PROCESSING FRAMES: {max_frames}')

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out    = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))

    vehicle_tracks  = {}
    next_vehicle_id = 0
    all_detections  = []
    crop_count      = 0
    frame_count     = 0

    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            logger.info('VIDEO FINISHED.')
            break

        ts          = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        frame_lines = []

        def log_frame(message: str) -> None:
            logger.info(message)
            frame_lines.append(f'[{ts}] {message}')

        log_frame(f'================ FRAME {frame_count} ================')
        h, w = frame.shape[:2]
        annotated_frame = frame.copy()
        cv2.polylines(annotated_frame, [ROI_POINTS], True, (255, 0, 0), 3)

        # ── ROI crop & resize for inference ──────────────────
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [ROI_POINTS], 255)
        roi_frame = cv2.bitwise_and(frame, frame, mask=mask)
        roi_x, roi_y, roi_w, roi_h = cv2.boundingRect(ROI_POINTS)
        model_frame = roi_frame[roi_y:roi_y + roi_h, roi_x:roi_x + roi_w]
        model_h, model_w = model_frame.shape[:2]

        if cfg.inference_width and model_w > cfg.inference_width:
            scale        = cfg.inference_width / model_w
            infer_w      = cfg.inference_width
            infer_h      = max(1, int(round(model_h * scale)))
            inference_frame = cv2.resize(model_frame, (infer_w, infer_h), interpolation=cv2.INTER_AREA)
            scale_x, scale_y = model_w / infer_w, model_h / infer_h
        else:
            inference_frame = model_frame
            infer_h, infer_w = model_h, model_w
            scale_x = scale_y = 1.0

        log_frame(f'ROI CROP ONLY: x={roi_x} y={roi_y} w={roi_w} h={roi_h}')
        log_frame(f'MODEL INPUT SIZE: {infer_w}x{infer_h}')

        # reset per-frame flags
        for track in vehicle_tracks.values():
            track['seen_this_frame']      = False
            track['frame_score']          = 0.0
            track['helmet_hit_this_frame'] = False

        # ── YOLO vehicle detection ────────────────────────────
        current_vehicle_ids    = []
        current_vehicle_id_set = set()
        yolo_results = yolo_model(inference_frame, verbose=False)
        for result in yolo_results:
            for box_index, yolo_box in enumerate(result.boxes):
                cls_id     = int(yolo_box.cls[0])
                conf       = float(yolo_box.conf[0])
                class_name = yolo_model.names[cls_id]
                log_frame(f'YOLO #{box_index}: {class_name} conf={conf:.3f}')
                if class_name not in {'motorcycle', 'bicycle'}:
                    continue

                bx1, by1, bx2, by2 = map(int, yolo_box.xyxy[0])
                x1 = int(round((bx1 * scale_x) + roi_x))
                y1 = int(round((by1 * scale_y) + roi_y))
                x2 = int(round((bx2 * scale_x) + roi_x))
                y2 = int(round((by2 * scale_y) + roi_y))
                box_h_px   = max(1, y2 - y1)
                expanded_y1 = max(0, int(round(y1 - cfg.vehicle_upward_expand * box_h_px)))
                vehicle_box = clamp_box([x1, expanded_y1, x2, y2], w, h)

                track_id = match_vehicle_track(vehicle_box, vehicle_tracks, cfg.vehicle_match_distance)
                if track_id is None:
                    track_id = next_vehicle_id
                    next_vehicle_id += 1
                    vehicle_tracks[track_id] = {
                        'box':                  vehicle_box,
                        'class':                class_name,
                        'best_score':           conf,
                        'helmet_streak':        0,
                        'confirmed_helmet':     False,
                        'misses':               0,
                        'first_frame':          frame_count,
                        'last_frame':           frame_count,
                        'seen_this_frame':      False,
                        'frame_score':          0.0,
                        'helmet_hit_this_frame': False,
                    }
                    log_frame(f'VEHICLE TRACK {track_id}: CREATED class={class_name}')
                else:
                    if vehicle_tracks[track_id]['seen_this_frame'] and conf <= vehicle_tracks[track_id]['frame_score']:
                        log_frame(f'VEHICLE TRACK {track_id}: SUPPRESSED DUPLICATE YOLO BOX class={class_name} conf={conf:.3f}')
                        continue
                    vehicle_tracks[track_id]['box']        = vehicle_box
                    vehicle_tracks[track_id]['class']      = class_name
                    vehicle_tracks[track_id]['best_score'] = max(vehicle_tracks[track_id]['best_score'], conf)
                    vehicle_tracks[track_id]['last_frame'] = frame_count
                    log_frame(f'VEHICLE TRACK {track_id}: UPDATED class={class_name}')

                vehicle_tracks[track_id]['seen_this_frame'] = True
                vehicle_tracks[track_id]['frame_score']     = conf
                vehicle_tracks[track_id]['misses']          = 0
                if track_id not in current_vehicle_id_set:
                    current_vehicle_ids.append(track_id)
                    current_vehicle_id_set.add(track_id)
                log_frame(f'VEHICLE TRACK {track_id} BOX: {vehicle_box}')

        # ── GroundingDINO helmet detection ────────────────────
        rgb_frame = cv2.cvtColor(inference_frame, cv2.COLOR_BGR2RGB)
        pil_frame = Image.fromarray(rgb_frame)
        inputs    = processor(images=pil_frame, text=cfg.prompt, return_tensors='pt')
        input_ids = inputs.input_ids.to(cfg.device)
        inputs    = to_device(inputs, cfg.device)

        with torch.no_grad():
            outputs = model(**inputs)

        results = post_process_groundingdino(
            processor, outputs, input_ids,
            torch.tensor([[infer_h, infer_w]], device=cfg.device),
            cfg.box_threshold, cfg.text_threshold,
        )

        accepted_helmet_boxes = []
        helmet_candidates     = []
        scores = results.get('scores', [])
        labels = results.get('labels', [])
        boxes  = results.get('boxes',  [])
        log_frame(f'GROUNDINGDINO RAW DETECTIONS: {len(scores)}')

        for box_index, (score, label, box) in enumerate(zip(scores, labels, boxes)):
            score_value = float(score.item() if hasattr(score, 'item') else score)
            class_name  = str(label.item() if hasattr(label, 'item') else label)
            bx1, by1, bx2, by2 = box.tolist()
            x1 = int(round((bx1 * scale_x) + roi_x))
            y1 = int(round((by1 * scale_y) + roi_y))
            x2 = int(round((bx2 * scale_x) + roi_x))
            y2 = int(round((by2 * scale_y) + roi_y))
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

            log_frame(
                f'DINO #{box_index}: label={class_name} conf={score_value:.3f} '
                f'box=({x1},{y1},{x2},{y2}) center=({cx},{cy})'
            )

            # ── basic validity filters ────────────────────────
            if 'helmet' not in class_name.lower():
                log_frame('  SKIPPED -> NOT HELMET')
                continue
            if cv2.pointPolygonTest(ROI_POINTS, (cx, cy), False) < 0:
                log_frame('  SKIPPED -> OUTSIDE ROI')
                continue
            box_w, box_h_px = x2 - x1, y2 - y1
            if box_w <= 0 or box_h_px <= 0:
                log_frame('  SKIPPED -> INVALID BOX')
                continue
            aspect = box_w / box_h_px
            if not (cfg.min_box_size <= box_w <= cfg.max_box_size):
                log_frame(f'  SKIPPED -> WIDTH OUTLIER w={box_w}')
                continue
            if not (cfg.min_box_size <= box_h_px <= cfg.max_box_size):
                log_frame(f'  SKIPPED -> HEIGHT OUTLIER h={box_h_px}')
                continue
            if not (cfg.min_aspect <= aspect <= cfg.max_aspect):
                log_frame(f'  SKIPPED -> ASPECT OUTLIER aspect={aspect:.2f}')
                continue

            helmet_box = clamp_box([x1, y1, x2, y2], w, h)

            # ── FIX 3: search ALL live tracks, not just current_vehicle_ids ──
            # This recovers helmets when YOLO drops the bike for 1-2 frames
            # but the track is still alive within vehicle_max_misses budget.
            candidate_track_ids = list(current_vehicle_ids)  # freshly seen first
            for tid in vehicle_tracks:
                if tid not in current_vehicle_id_set:
                    candidate_track_ids.append(tid)  # still-alive tracks appended

            matched_vehicle_id       = None
            matched_vehicle_distance = None
            match_reason             = ''

            for track_id in candidate_track_ids:
                vx1, vy1, vx2, vy2 = vehicle_tracks[track_id]['box']

                # ── FIX 1: strict inside OR pillion margin above ──────────
                inside   = vx1 <= cx <= vx2 and vy1 <= cy <= vy2
                pillion  = vx1 <= cx <= vx2 and (vy1 - cfg.pillion_margin_above) <= cy < vy1

                if inside or pillion:
                    tx, ty   = box_center(vehicle_tracks[track_id]['box'])
                    distance = float(np.hypot(cx - tx, cy - ty))
                    if matched_vehicle_distance is None or distance < matched_vehicle_distance:
                        matched_vehicle_id       = track_id
                        matched_vehicle_distance = distance
                        match_reason             = 'PILLION' if pillion else 'INSIDE'

            if matched_vehicle_id is None:
                log_frame('  SKIPPED -> HELMET NOT INSIDE OR ABOVE TWO-WHEELER BOX')
                continue

            helmet_candidates.append({
                'box':              helmet_box,
                'score':            score_value,
                'vehicle_track_id': matched_vehicle_id,
                'box_index':        box_index,
                'match_reason':     match_reason,
            })
            log_frame(f'  CANDIDATE HELMET ({match_reason}) FOR VEHICLE TRACK {matched_vehicle_id}')

        # ── FIX 2: keep top-N helmets per vehicle track ───────
        # Sort all candidates per track by score descending, keep max_helmets_per_vehicle.
        helmets_by_vehicle = {}
        for candidate in helmet_candidates:
            tid = candidate['vehicle_track_id']
            helmets_by_vehicle.setdefault(tid, []).append(candidate)

        best_helmets_by_vehicle = {}
        for tid, candidates in helmets_by_vehicle.items():
            candidates.sort(key=lambda c: c['score'], reverse=True)
            kept      = candidates[:cfg.max_helmets_per_vehicle]
            suppressed = candidates[cfg.max_helmets_per_vehicle:]
            best_helmets_by_vehicle[tid] = kept
            for sup in suppressed:
                log_frame(
                    f"  SUPPRESSED EXTRA HELMET DINO #{sup['box_index']} "
                    f"for vehicle track {tid}; score={sup['score']:.3f} (beyond max {cfg.max_helmets_per_vehicle})"
                )

        # ── process accepted helmets ──────────────────────────
        for track_id, kept_candidates in best_helmets_by_vehicle.items():
            track = vehicle_tracks[track_id]
            track['helmet_hit_this_frame'] = True
            track['helmet_streak'] += 1
            if track['helmet_streak'] >= cfg.helmet_confirm_frames:
                track['confirmed_helmet'] = True
            log_frame(
                f"  HELMET(S) FOR VEHICLE TRACK {track_id}: count={len(kept_candidates)} "
                f"streak={track['helmet_streak']}/{cfg.helmet_confirm_frames} "
                f"confirmed={track['confirmed_helmet']}"
            )

            for candidate in kept_candidates:
                helmet_box  = candidate['box']
                score_value = candidate['score']
                path = save_crop(frame, helmet_box, helmet_crop_dir, 'helmet', frame_count, score_value, ts)
                if path:
                    crop_count += 1
                accepted_helmet_boxes.append((helmet_box, score_value, track_id, path))
                all_detections.append({
                    'frame':            frame_count,
                    'timestamp':        ts,
                    'label':            'helmet',
                    'score':            score_value,
                    'box':              helmet_box,
                    'vehicle_track_id': track_id,
                    'match_reason':     candidate['match_reason'],
                    'crop_path':        str(path) if path else None,
                })

        # ── streak reset for vehicles with no helmet hit ──────
        for track_id in current_vehicle_ids:
            track = vehicle_tracks[track_id]
            if not track['helmet_hit_this_frame'] and not track['confirmed_helmet']:
                track['helmet_streak'] = 0
                log_frame(f'VEHICLE TRACK {track_id}: helmet streak reset before confirmation')

        # ── retire stale tracks ───────────────────────────────
        retired_ids = []
        for track_id, track in vehicle_tracks.items():
            if not track['seen_this_frame']:
                track['misses'] += 1
            if track['misses'] > cfg.vehicle_max_misses:
                retired_ids.append(track_id)
        for track_id in retired_ids:
            log_frame(f"VEHICLE TRACK {track_id}: RETIRED after misses={vehicle_tracks[track_id]['misses']}")
            del vehicle_tracks[track_id]

        # ── draw vehicle annotations ──────────────────────────
        frame_helmet_count    = 0
        frame_no_helmet_count = 0
        for track_id in current_vehicle_ids:
            track = vehicle_tracks.get(track_id)
            if track is None:
                continue
            x1, y1, x2, y2 = track['box']
            if track['confirmed_helmet']:
                status = 'helmet'
                color  = (0, 255, 0)
                frame_helmet_count += 1
            else:
                status = 'no_helmet'
                color  = (0, 0, 255)
                frame_no_helmet_count += 1
                path = save_crop(frame, track['box'], no_helmet_crop_dir, 'no_helmet', frame_count, track['best_score'], ts)
                if path:
                    crop_count += 1
                all_detections.append({
                    'frame':            frame_count,
                    'timestamp':        ts,
                    'label':            'no_helmet',
                    'score':            track['best_score'],
                    'box':              track['box'],
                    'vehicle_track_id': track_id,
                    'crop_path':        str(path) if path else None,
                    'reason':           'vehicle has no confirmed helmet streak',
                })

            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 4)
            cv2.putText(
                annotated_frame, f'{status} T{track_id}',
                (x1, max(30, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3,
            )
            log_frame(
                f"VEHICLE TRACK {track_id} FINAL: {status} "
                f"confirmed={track['confirmed_helmet']} streak={track['helmet_streak']} box={track['box']}"
            )

        # ── draw helmet box annotations ───────────────────────
        for helmet_box, score_value, matched_vehicle_id, _ in accepted_helmet_boxes:
            if matched_vehicle_id is not None and vehicle_tracks.get(matched_vehicle_id, {}).get('confirmed_helmet'):
                x1, y1, x2, y2 = helmet_box
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
                cv2.putText(
                    annotated_frame, f'helmet {score_value:.2f}',
                    (x1, max(30, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2,
                )

        log_frame(
            f'FRAME {frame_count} SUMMARY | vehicles={len(current_vehicle_ids)} '
            f'helmet_vehicles={frame_helmet_count} no_helmet_vehicles={frame_no_helmet_count} '
            f'accepted_helmet_boxes={len(accepted_helmet_boxes)} total_crops={crop_count}'
        )

        (frame_log_dir / f'frame_{frame_count:06d}_debug.txt').write_text(
            '\n'.join(frame_lines) + '\n', encoding='utf-8'
        )
        out.write(annotated_frame)
        frame_count += 1

        if frame_count % 10 == 0:
            logger.info(f'PROCESSED {frame_count}/{max_frames} FRAMES')

    cap.release()
    out.release()
    detections_json_path.write_text(json.dumps(all_detections, indent=2), encoding='utf-8')

    helmet_events    = sum(1 for d in all_detections if d['label'] == 'helmet')
    no_helmet_events = sum(1 for d in all_detections if d['label'] == 'no_helmet')
    pillion_hits     = sum(1 for d in all_detections if d.get('match_reason') == 'PILLION')

    logger.info('========== RUN COMPLETE ==========')
    logger.info(f'FRAMES PROCESSED        : {frame_count}')
    logger.info(f'HELMET DETECTIONS       : {helmet_events}')
    logger.info(f'  of which PILLION hits : {pillion_hits}')
    logger.info(f'NO_HELMET DETECTIONS    : {no_helmet_events}')
    logger.info(f'OUTPUT VIDEO            : {output_video_path}')
    logger.info(f'DEBUG LOG               : {debug_log_path}')
    logger.info(f'FRAME LOG DIR           : {frame_log_dir}')
    logger.info(f'DETECTIONS JSON         : {detections_json_path}')

    return {
        'frames_processed':    frame_count,
        'helmet_events':       helmet_events,
        'pillion_hits':        pillion_hits,
        'no_helmet_events':    no_helmet_events,
        'output_video_path':   str(output_video_path),
        'debug_log_path':      str(debug_log_path),
        'frame_log_dir':       str(frame_log_dir),
        'detections_json_path':str(detections_json_path),
    }


# ── run ──────────────────────────────────────────────────────
if not config.input_video.exists():
    raise FileNotFoundError(
        f'Video not found: {config.input_video}. Update INPUT_VIDEO at the top of this cell.'
    )

results = run_inference(config)
results


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.2 MB/s eta 0:00:00
Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Config(input_video=PosixPath('/content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset/laketown_TG_gholaghata_service_road_1min.avi'), output_dir=PosixPath('/content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset/helmet_vehicle_roi_outputs'), groundingdino_model_id='IDEA-Research/grounding-dino-tiny', yolo_model='yolov8s.pt', prompt='helmet.', box_threshold=0.5, text_threshold=0.3, inference_width=896, min_box_size=20, max_box_size=180, min_aspect=0.35, max_aspect=2.5, helmet_confirm_frames=2, vehicle_max_misses=20,

[2026-05-19 09:48:49] ========== HELMET + TWO-WHEELER ROI INFERENCE (PILLION FIX) ==========
INFO:helmet_vehicle_roi:========== HELMET + TWO-WHEELER ROI INFERENCE (PILLION FIX) ==========
[2026-05-19 09:48:49] INPUT VIDEO : /content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset/laketown_TG_gholaghata_service_road_1min.avi
INFO:helmet_vehicle_roi:INPUT VIDEO : /content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset/laketown_TG_gholaghata_service_road_1min.avi
[2026-05-19 09:48:49] OUTPUT DIR  : /content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset/helmet_vehicle_roi_outputs
INFO:helmet_vehicle_roi:OUTPUT DIR  : /content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset/helmet_vehicle_roi_outputs
[2026-05-19 09:48:49] DEVICE      : cuda
INFO:helmet_vehicle_roi:DEVICE      : cuda
[2026-05-19 09:48:49] DINO MODEL  : IDEA-Research/grounding-dino-tiny
INFO:helme

preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/689M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

[2026-05-19 09:50:55] Loading YOLO...
INFO:helmet_vehicle_roi:Loading YOLO...


Streaming output truncated to the last 5000 lines.
[2026-05-19 10:01:40] YOLO #5: person conf=0.352
INFO:helmet_vehicle_roi:YOLO #5: person conf=0.352
[2026-05-19 10:01:40] YOLO #6: person conf=0.311
INFO:helmet_vehicle_roi:YOLO #6: person conf=0.311
[2026-05-19 10:01:40] YOLO #7: motorcycle conf=0.272
INFO:helmet_vehicle_roi:YOLO #7: motorcycle conf=0.272
[2026-05-19 10:01:40] VEHICLE TRACK 22: SUPPRESSED DUPLICATE YOLO BOX class=motorcycle conf=0.272
INFO:helmet_vehicle_roi:VEHICLE TRACK 22: SUPPRESSED DUPLICATE YOLO BOX class=motorcycle conf=0.272
[2026-05-19 10:01:40] YOLO #8: motorcycle conf=0.261
INFO:helmet_vehicle_roi:YOLO #8: motorcycle conf=0.261
[2026-05-19 10:01:40] VEHICLE TRACK 23: SUPPRESSED DUPLICATE YOLO BOX class=motorcycle conf=0.261
INFO:helmet_vehicle_roi:VEHICLE TRACK 23: SUPPRESSED DUPLICATE YOLO BOX class=motorcycle conf=0.261
[2026-05-19 10:01:40] GROUNDINGDINO RAW DETECTIONS: 2
INFO:helmet_vehicle_roi:GROUNDINGDINO RAW DETECTIONS: 2
[2026-05-19 10:01:40] DINO 

{'frames_processed': 1500,
 'helmet_events': 326,
 'pillion_hits': 45,
 'no_helmet_events': 417,
 'output_video_path': '/content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset/helmet_vehicle_roi_outputs/output_laketown_TG_gholaghata_service_road_1min.avi',
 'debug_log_path': '/content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset/helmet_vehicle_roi_outputs/debug_log.txt',
 'frame_log_dir': '/content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset/helmet_vehicle_roi_outputs/frame_debug_logs',
 'detections_json_path': '/content/drive/MyDrive/Computer_Vision_Projects_Research/indian_helmet_detection_dataset/helmet_vehicle_roi_outputs/detections.json'}